# 1. Tissues-Specific Gene Expression

## 1.1 Reactome (Genes)

In [ ]:
from data.pathways.reactome import ReactomeNetwork

In [ ]:
R = ReactomeNetwork()

In [ ]:
R.reactome.pathway_genes

In [ ]:
reactome_gene = R.reactome.pathway_genes.gene.unique()

In [ ]:
len(reactome_gene)

In [ ]:
R.ge

## 1.2 FANTOM (Genes & Tisseus)

### 1.2.1 Read Data

In [ ]:
import pandas as pd 
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np 

In [ ]:
fantom = pd.read_csv("_database/genes/FantomCAT_robust_genes_vs_traits_pvalue_nonzero_04072021.csv",header=1)

In [ ]:
fantom.head()

In [ ]:
fantom.describe()

In [ ]:
# value_counts of RNAs type
fantom.geneClass.value_counts()
fantom_gene = fantom.geneName.values

In [ ]:
fantom_ex = fantom.iloc[:, 10:] # fantom_ex = FANTOM gene EXpression matrix

fantom_ex_cod = fantom_ex[fantom.geneClass == "coding_mRNA"] # fantom_ex_cod = FANTOM gene EXpression in CODing rna 
fantom_ex_bin = fantom_ex < 0.05 # fantom_ex_bin = FANTOM gene EXpression in BINary if p-value > 0.05 
fantom_ex_log = -np.log(fantom_ex) # fantom_ex_log = FANTOM gene EXpression in LOGarithm 
fantom_ex_sig = fantom_ex[fantom_ex_bin.sum(axis=1) > 0] # fantom_ex_sig = FANTOM gene EXpression which are SIGnificant(p-values < 0.05)

fantom_ex_cod_bin = fantom_ex_cod < 0.05 # fantom_ex_cod_bin = FANTOM gene EXpression in CODing rna in BINary if p-value > 0.05 
fantom_ex_cod_log = -np.log(fantom_ex_cod) # fantom_ex_cod_log = FANTOM gene EXpression in CODing rna in LOGarithm 
fantom_ex_cod_sig = fantom_ex_cod[fantom_ex_cod_bin.sum(axis=1) > 0] # fantom_ex_cod_sig = FANTOM gene EXpression in CODing rna which are SIGnificant(p-values < 0.05)

fantom_ex_sig_bin = fantom_ex_sig < 0.05 # fantom_ex_bin_sig = FANTOM gene EXpression in BINary which are SIGnificant(p-values < 0.05)
fantom_ex_sig_log = -np.log(fantom_ex_sig) # fantom_ex_log_sig = FANTOM gene EXpression in LOGarithm  which are SIGnificant(p-values < 0.05)

fantom_ex_cod_sig_log = -np.log(fantom_ex_cod_sig)
fantom_ex_cod_sig_bin = fantom_ex_cod_sig < 0.05

In [ ]:
# plot of Number of tissues with p-values < 0.05
temp = fantom_ex_bin.sum(axis=1).value_counts().sort_index()
temp.plot(kind="bar",
          xlabel="Number of tissues with p-values < 0.05",
          ylabel="Number of genes")
plt.show()

In [ ]:
# plot of means of p-values in each gene distribution 
fantom_ex.mean(axis=1).hist(bins=50)

In [ ]:
# plot of standard deviation of p-values in each gene distribution 
fantom_ex.std(axis=1).hist(bins=50)

* Note: Means may not be a good choice because you may find some genes that have mean p-values < 1 or pre-defined threshold but are not significant(p-value<0.05)

In [ ]:
# there are 9960 genes with mean of p-values < 1 but are not significant
sum((fantom_ex.mean(axis=1)<1) * (fantom_ex_bin.sum(axis=1) == 0))

### 1.2.2 K-means clustering 

In [ ]:
from sklearn.cluster import KMeans

In [ ]:
num_clusters = 20

kmeans = KMeans(n_clusters=num_clusters)
clusters = kmeans.fit_predict(fantom_ex_bin.T)

In [ ]:
import matplotlib.pyplot as plt
 
# Create a range of values for k
k_range = range(1, 25)
 
# Initialize an empty list to
# store the inertia values for each k
inertia_values = []
 
# Fit and plot the data for each k value
for k in k_range:
    kmeans = KMeans(n_clusters=k, \
                    init='k-means++', random_state=42)
    y_kmeans = kmeans.fit_predict(fantom_ex.T)
    inertia_values.append(kmeans.inertia_)

# Plot the inertia values for each k
plt.plot(k_range, inertia_values, 'bo-')
plt.title('Elbow Method')
plt.xlabel('Number of clusters (k)')
plt.ylabel('Inertia')
plt.show()

### 1.2.3 PCA

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

In [ ]:
# standardized_data = StandardScaler().fit_transform(fantom_ex_sig_bin.T)
pca = PCA(n_components=fantom_ex_sig_bin.shape[1]-1)
pca_data = pca.fit_transform(fantom_ex_sig_bin.T)
np.cumsum(pca.explained_variance_ratio_)

In [ ]:
plt.scatter(pca_data[:, 0], pca_data[:, 1],c=clusters)
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.title("PCA Result")
plt.show()

In [ ]:
fantom_PCA_df = pd.DataFrame(data=pca_data,index=fantom_ex.columns)

In [ ]:
fantom_PCA_df

### 1.2.4 TSNE

In [ ]:
from sklearn.manifold import TSNE

tsne = TSNE(n_components=2, verbose=1, perplexity=20, n_iter=500)
tsne_results = tsne.fit_transform(fantom_ex_bin.T)

In [ ]:
plt.scatter(tsne_results[:, 0], tsne_results[:, 1],c=clusters)
plt.xlabel("TSNE Component 1")
plt.ylabel("TSNE Component 2")
plt.title("TSNE Result")
plt.show()

### 1.2.5 Heatmap

In [ ]:
# Create the heatmap
plt.figure(figsize=(10, 6))
sns.clustermap(fantom_PCA_df.iloc[:50,:10], center=3, fmt=".5f", linewidths=0.5)
plt.title("Gene Expression Heatmap")
plt.xlabel("Samples")
plt.ylabel("Genes")
plt.show()

## 1.3 GTEx

### 1.3.1 Read Data 

In [ ]:
# download data from https://storage.googleapis.com/gtex_analysis_v8/rna_seq_data/GTEx_Analysis_2017-06-05_v8_RNASeQCv1.1.9_gene_median_tpm.gct.gz
gtex = pd.read_csv("_database/genes/GTEx_Analysis_2017-06-05_v8_RNASeQCv1.1.9_gene_median_tpm.gct",sep="\t")

In [ ]:
gtex.head()

In [ ]:
gtex_gene = gtex.Description.values

In [ ]:
gtex.describe()

In [ ]:
# from column index=2 gene expression starts 
gtex.columns[:23]

In [ ]:
gtex_ex = gtex.iloc[:,2:]

### 1.3.2 K-means clustering

In [ ]:
from sklearn.cluster import KMeans
num_clusters = 10

kmeans = KMeans(n_clusters=num_clusters)
clusters = kmeans.fit_predict(gtex_ex.T)

### 1.3.3 PCA

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

In [ ]:
standardized_data = StandardScaler().fit_transform(gtex_ex.T)
pca = PCA(n_components=standardized_data.shape[0]-1)
pca_data = pca.fit_transform(standardized_data)
np.cumsum(pca.explained_variance_ratio_)

In [ ]:
plt.scatter(pca_data[:, 0], pca_data[:, 1],c=clusters)
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.title("PCA Result")
plt.show()

In [ ]:
fantom_PCA_df = pd.DataFrame(data=pca_data,index=fantom_ex.columns)

In [ ]:
fantom_PCA_df

In [ ]:
gtex_gene = gtex.Description.unique()
reactome_gtex_gene = np.intersect1d(reactome_gene, gtex_gene)
len(reactome_gtex_gene)

In [ ]:
# reactome and gtex intersection genes expression
reactome_gtex_index = [i in reactome_gtex_gene for i in gtex.Description]
reactome_gtex_df = gtex[reactome_gtex_index]

## 1.4 Comparison of Genes

In [ ]:
# import modules
from matplotlib_venn import venn3 
from matplotlib import pyplot as plt

In [ ]:
len(np.intersect1d(reactome_gene,gtex_gene))

In [ ]:
venn = venn3([set(reactome_gene), set(fantom_gene), set(gtex_gene)],
             set_labels=("Reactome", "Fantom","GTEx"))
plt.show()

# 2. Data 

In [1]:
import numpy as np
import importlib
from data.data_access import Data
from preprocessing import pre
params_file = 'train/params/P1000/pnet/onsplit_average_reg_10_tanh_large_testing.py'

In [2]:
loader = importlib.machinery.SourceFileLoader('params', params_file)
params = loader.load_module()

In [3]:
temp = Data(**params.data[0])

In [4]:
x, y, inf, col = temp.get_data()

In [5]:
x_train, x_validate_, x_test_, y_train, y_validate_, y_test_, info_train, info_validate_, info_test_, cols = temp.get_train_validate_test()

In [6]:
import pandas as pd 
tcga = pd.read_csv("_database/genes/tcga_prostate_expressed_genes_and_cancer_genes.csv")
hugo = pd.read_csv("_database/genes/HUGO_genes/protein-coding_gene_with_coordinate_minimal.txt",header=None,sep="\t")
hugo = hugo.iloc[:,3]
tcga = tcga.genes
from data.prostate_paper.data_reader import load_data_type
mut = load_data_type("mut_important", cnv_levels=3,
                     cnv_filter_single_event=True,
                     mut_binary=True, selected_genes=None)
mut = mut[0]
print("mut.shape: ",mut.shape)
cnv_amp = load_data_type("cnv_amp", cnv_levels=3,
                     cnv_filter_single_event=True,
                     mut_binary=True, selected_genes=None)
cnv_amp = cnv_amp[0]
print("cnv_amp.shape",cnv_amp.shape)
cnv_del = load_data_type("cnv_del", cnv_levels=3,
                     cnv_filter_single_event=True,
                     mut_binary=True, selected_genes=None)
cnv_del = cnv_del[0]
print("cnv_del.shape",cnv_del.shape)

mut_cnv = mut.columns.union(cnv_amp.columns)
mut_cnv_tcga = mut_cnv.intersection(tcga)
mut_cnv_tcga_hugo = mut_cnv_tcga.intersection(hugo)
len(mut_cnv_tcga_hugo)

mut.shape:  (1011, 14378)
cnv_amp.shape (1013, 13802)
cnv_del.shape (1013, 13802)


9229

In [7]:
X = np.concatenate((x_test_, x_validate_, x_train))
Y = np.concatenate((y_test_, y_validate_, y_train))

In [8]:
x_t = np.concatenate((x_test_, x_validate_))
y_t = np.concatenate((y_test_, y_validate_))
info_t = info_test_.append(info_validate_)

In [9]:
print("x_train:", x_train.shape)
print("x_train:", y_train.shape)

print("x_validate_:", x_validate_.shape)
print("y_validate_:", y_validate_.shape)

print("x_test:", x_t.shape)
print("y_test:", y_t.shape)

print("columns:", len(cols))

x_train: (807, 27687)
x_train: (807, 1)
x_validate_: (102, 27687)
y_validate_: (102, 1)
x_test: (204, 27687)
y_test: (204, 1)
columns: 27687


In [10]:
proc = pre.get_processor(params.pre)
if proc:
    proc.fit(x_train)
    x_train = proc.transform(x_train)
    x_test = proc.transform(x_t)
    if scipy.sparse.issparse(x_train):
        x_train = x_train.todense()
        x_test = x_test.todense()

{'type': None}


# 3. Model

In [11]:
from copy import deepcopy
from model import nn

## 3.1 Trainig 

* Note: Change the `single_outputs` argument in `onsplit_average_reg_10_tanh_large_testing.py` file to True to make single node output
        

In [12]:
model_params_ = deepcopy(params.models[0])

In [13]:
model = nn.Model(**model_params_['params'])

In [14]:
history = model.fit(x_train, y_train)

{'id': 'ALL', 'type': 'prostate_paper', 'params': {'data_type': ['mut_important', 'cnv_del', 'cnv_amp'], 'drop_AR': False, 'cnv_levels': 3, 'mut_binary': True, 'balanced_data': False, 'combine_type': 'union', 'use_coding_genes_only': True, 'selected_genes': 'tcga_prostate_expressed_genes_and_cancer_genes.csv', 'training_split': 0}}
n_hidden_layers 5


(1011, 27687)
(1011, 1)
(1011,)
(27687,)
input dimension 27687 self.units 9229
n_inputs_per_node 3.0
self.kernel_initializer <keras.src.regularizers.L2 object at 0x2a2dd6e90> <keras.src.initializers.initializers.LecunUniform object at 0x2a4281190> <keras.src.regularizers.L2 object at 0x2a2dd6e90>
input dimensions (None, 27687)


2023-09-01 20:42:19.261457: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M1 Max
2023-09-01 20:42:19.261485: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 64.00 GB
2023-09-01 20:42:19.261490: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 24.00 GB
2023-09-01 20:42:19.261523: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:303] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2023-09-01 20:42:19.261541: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:269] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


layer # 0
pathways 1387
genes 9275
filtered_map (9229, 0)
filtered_map (9229, 0)
filtered_map (9229, 0)
layer # 1
pathways 1066
genes 1399
filtered_map (1387, 0)
filtered_map (1387, 0)
filtered_map (1387, 0)
layer # 2
pathways 447
genes 1068
filtered_map (1066, 0)
filtered_map (1066, 0)
filtered_map (1066, 0)
layer # 3
pathways 147
genes 448
filtered_map (447, 0)
filtered_map (447, 0)
filtered_map (447, 0)
layer # 4
pathways 26
genes 147
filtered_map (147, 0)
filtered_map (147, 0)
filtered_map (147, 0)
layer # 5
pathways 1
genes 26
filtered_map (26, 0)
filtered_map (26, 0)
filtered_map (26, 0)
original dropout [0.5, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1]
dropout [1, 2, 3, 4, 5] [0.5, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1] [0.001, 0.001, 0.001, 0.001, 0.001, 0.001, 0.001]
4749 15651.0
1387 1396.0
1066 1070.0
447 447.0
147 148.0
26 26.0
map is:
maps[0] 9229
maps[1] 1387
maps[2] 1066
maps[3] 447
maps[4] 147
maps[5] 26
               root
R-HSA-109582    1.0
R-HSA-1474244   1.0
R-HSA-112316    1.0
R-HSA-143

2023-09-01 20:42:25.983991: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:114] Plugin optimizer for device_type GPU is enabled.


...


Epoch 157: LearningRateScheduler setting learning rate to 1.5625e-05.
Epoch 157/300
9/9 - 0s - loss: 0.5316 - f1: 0.9554 - lr: 1.5625e-05 - 374ms/epoch - 42ms/step

Epoch 158: LearningRateScheduler setting learning rate to 1.5625e-05.
Epoch 158/300
9/9 - 0s - loss: 0.5315 - f1: 0.9438 - lr: 1.5625e-05 - 362ms/epoch - 40ms/step

Epoch 159: LearningRateScheduler setting learning rate to 1.5625e-05.
Epoch 159/300
9/9 - 0s - loss: 0.5314 - f1: 0.9364 - lr: 1.5625e-05 - 359ms/epoch - 40ms/step

Epoch 160: LearningRateScheduler setting learning rate to 1.5625e-05.
Epoch 160/300
9/9 - 0s - loss: 0.5314 - f1: 0.9554 - lr: 1.5625e-05 - 362ms/epoch - 40ms/step

Epoch 161: LearningRateScheduler setting learning rate to 1.5625e-05.
Epoch 161/300
9/9 - 0s - loss: 0.5313 - f1: 0.9429 - lr: 1.5625e-05 - 370ms/epoch - 41ms/step

Epoch 162: LearningRateScheduler setting learning rate to 1.5625e-05.
Epoch 162/300
9/9 - 0s - loss: 0.5312 - f1: 0.9556 - lr: 1.5625e-05 - 368ms/epoch - 41ms/step

Epoch 163


Epoch 207: LearningRateScheduler setting learning rate to 3.90625e-06.
Epoch 207/300
9/9 - 0s - loss: 0.5284 - f1: 0.9581 - lr: 3.9063e-06 - 360ms/epoch - 40ms/step

Epoch 208: LearningRateScheduler setting learning rate to 3.90625e-06.
Epoch 208/300
9/9 - 0s - loss: 0.5284 - f1: 0.9595 - lr: 3.9063e-06 - 369ms/epoch - 41ms/step

Epoch 209: LearningRateScheduler setting learning rate to 3.90625e-06.
Epoch 209/300
9/9 - 0s - loss: 0.5284 - f1: 0.9414 - lr: 3.9063e-06 - 413ms/epoch - 46ms/step

Epoch 210: LearningRateScheduler setting learning rate to 3.90625e-06.
Epoch 210/300
9/9 - 0s - loss: 0.5284 - f1: 0.9571 - lr: 3.9063e-06 - 420ms/epoch - 47ms/step

Epoch 211: LearningRateScheduler setting learning rate to 3.90625e-06.
Epoch 211/300
9/9 - 0s - loss: 0.5284 - f1: 0.9478 - lr: 3.9063e-06 - 421ms/epoch - 47ms/step

Epoch 212: LearningRateScheduler setting learning rate to 3.90625e-06.
Epoch 212/300
9/9 - 0s - loss: 0.5283 - f1: 0.9184 - lr: 3.9063e-06 - 385ms/epoch - 43ms/step

Epo

Epoch 256/300
9/9 - 0s - loss: 0.5276 - f1: 0.9557 - lr: 9.7656e-07 - 386ms/epoch - 43ms/step

Epoch 257: LearningRateScheduler setting learning rate to 9.765625e-07.
Epoch 257/300
9/9 - 0s - loss: 0.5276 - f1: 0.8500 - lr: 9.7656e-07 - 401ms/epoch - 45ms/step

Epoch 258: LearningRateScheduler setting learning rate to 9.765625e-07.
Epoch 258/300
9/9 - 0s - loss: 0.5276 - f1: 0.9565 - lr: 9.7656e-07 - 400ms/epoch - 44ms/step

Epoch 259: LearningRateScheduler setting learning rate to 9.765625e-07.
Epoch 259/300
9/9 - 0s - loss: 0.5276 - f1: 0.9581 - lr: 9.7656e-07 - 385ms/epoch - 43ms/step

Epoch 260: LearningRateScheduler setting learning rate to 9.765625e-07.
Epoch 260/300
9/9 - 0s - loss: 0.5276 - f1: 0.9579 - lr: 9.7656e-07 - 373ms/epoch - 41ms/step

Epoch 261: LearningRateScheduler setting learning rate to 9.765625e-07.
Epoch 261/300
9/9 - 0s - loss: 0.5276 - f1: 0.9572 - lr: 9.7656e-07 - 371ms/epoch - 41ms/step

Epoch 262: LearningRateScheduler setting learning rate to 9.765625e-07

2023-09-01 20:44:24.219908: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:114] Plugin optimizer for device_type GPU is enabled.


(807, 1)
[0.1  0.11 0.12 0.13 0.14 0.15 0.16 0.17 0.18 0.19 0.2  0.21 0.22 0.23
 0.24 0.25 0.26 0.27 0.28 0.29 0.3  0.31 0.32 0.33 0.34 0.35 0.36 0.37
 0.38 0.39 0.4  0.41 0.42 0.43 0.44 0.45 0.46 0.47 0.48 0.49 0.5  0.51
 0.52 0.53 0.54 0.55 0.56 0.57 0.58 0.59 0.6  0.61 0.62 0.63 0.64 0.65
 0.66 0.67 0.68 0.69 0.7  0.71 0.72 0.73 0.74 0.75 0.76 0.77 0.78 0.79
 0.8  0.81 0.82 0.83 0.84 0.85 0.86 0.87 0.88 0.89]
    accuracy  precision        f1    recall    th
0   0.583643   0.441860  0.612903  1.000000  0.10
1   0.615861   0.461806  0.631829  1.000000  0.11
2   0.645601   0.481884  0.650367  1.000000  0.12
3   0.688971   0.514507  0.679438  1.000000  0.13
4   0.749690   0.568376  0.724796  1.000000  0.14
..       ...        ...       ...       ...   ...
75  0.815366   1.000000  0.610966  0.439850  0.85
76  0.805452   1.000000  0.581333  0.409774  0.86
77  0.794300   1.000000  0.546448  0.375940  0.87
78  0.786865   1.000000  0.522222  0.353383  0.88
79  0.775713   1.000000  0.484330 

In [15]:
test = model.predict(x_t)

7/7 [==============================] - 0s 13ms/step
(204, 1)


In [16]:
res = y_t == test

In [17]:
res.sum()/len(y_t)

0.7941176470588235

## 3.2 Cross-Validation

In [ ]:
from sklearn.model_selection import KFold
from statistics import mean, stdev

In [ ]:
model_params_ = deepcopy(params.models[0])

In [ ]:
n_splits = 5  # Number of folds
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

In [ ]:
accurasies = []
for train_indices, val_indices in kf.split(X):
    x_train_fold, x_val_fold = X[train_indices], X[val_indices]
    y_train_fold, y_val_fold = Y[train_indices], Y[val_indices]
    
    model = nn.Model(**model_params_['params'])
    
    model.fit(x_train_fold, y_train_fold)
    test = model.predict(x_val_fold)
    res = y_val_fold == test
    accurasies.append(res.sum()/len(y_val_fold))
    print("Accurasies: ", accurasies)

In [ ]:
accurasies

In [ ]:
print(mean(accurasies)) 
print(stdev(accurasies)) 
# Single Node : 0.8338243183924303 / Multiple Node : 0.8348241720723797
# Single Node : 0.014730575250953129 / Multiple Node : 0.033270867867846915